# KPI Analysis – Vanguard A/B Test

This notebook builds on the prepared client-level table created from:

1. Demographics / client profile data
2. Experiment roster data
3. Aggregated web behavior data

The goal is to validate the final analysis dataset and calculate the selected KPIs for the A/B test comparison between Test and Control groups.

## Selected KPIs

### Mandatory KPIs
- Completion Rate
- Time to Completion
- Error Rate

### Additional KPIs
- Step Conversion Rate
- Sessions per Client
- Average Steps per Session


In [ ]:
import pandas as pd
import numpy as np

## 1. Load Final Client-Level Dataset

We use the final client-level table created in the data preparation notebooks.
We refer to this dataset as `df_clients` throughout the analysis.

This table should already contain:
- one row per client
- demographic information
- experiment group information
- aggregated web behavior features

In [ ]:
# Load final client-level dataset
df_clients = pd.read_csv("../data/clean/clientroster_clean.csv")

### Basic Data Validation

Before calculating KPIs, we verify that the dataset is suitable for analysis.

Key checks:
- One row per client
- Presence of Test and Control groups
- No missing values in key columns
- Plausibility of aggregated features

In [ ]:
df_clients.shape

In [ ]:
df_clients.head()

In [ ]:
df_clients.info()

In [ ]:
df_clients["client_id"].duplicated().sum()

In [ ]:
df_clients["Variation"].value_counts(dropna=False)

**Validation Summary:**

The dataset contains one row per client and includes both Test and Control groups.  
No critical data quality issues were identified, and the dataset is suitable for KPI analysis.

## 3. KPI Definitions

We define the KPIs used to evaluate the experiment.

### Primary KPI
- Completion Rate: % of users reaching the "confirm" step

### Secondary KPIs
- Time to Completion: Average time to complete the process (based on successful sessions)
- Error Rate: % of sessions with backward steps
- Step Conversion Rate: % of users moving from one step to the next
- Sessions per Client: Number of visits per client
- Avg Steps per Session: Average number of steps per session

### Time to Completion

The final client-level table does not contain session duration directly.

Therefore, we load the visit-level table and calculate the average duration of successful sessions per client.  
A successful session is defined as a visit where the user reached the `confirm` step.

This client-level time feature is then merged back into the main analysis dataset.

In [ ]:
df_visits = pd.read_csv("../data/clean/visits_clean.csv")


In [ ]:
df_visits.columns

We only consider successful sessions for this KPI, because "Time to Completion" should reflect how long it takes users to successfully complete the process.

In [ ]:
# Filter visits where users reached the final "confirm" step
df_successful_visits = df_visits[df_visits["reached_confirm"] == 1]

# Aggregate average time to completion per client
df_time_to_completion = (
    df_successful_visits
    .groupby("client_id")
    .agg(avg_time_to_completion=("visit_duration_sec", "mean"))
    .reset_index()
)

We merge the client-level time feature back into the main analysis dataset.
Clients without a successful session will have missing values for `avg_time_to_completion`, which is expected.

In [ ]:
# Merge average successful session duration into the main client-level dataset
df_clients = df_clients.merge(df_time_to_completion, on="client_id", how="left")

# Sanity check: distribution of the new time feature
df_clients["avg_time_to_completion"].describe()

### Time to Completion (KPI Calculation)

We now calculate the average time to completion for each group (Test vs Control).

Only clients with at least one successful session are considered, since the metric is defined for completed journeys only.

In [ ]:
# Re-create test and control groups (after merging new feature)
df_test = df_clients[df_clients["Variation"] == "Test"]
df_control = df_clients[df_clients["Variation"] == "Control"]

# Keep only clients with successful sessions (non-null time values)
df_test_completed = df_test[df_test["avg_time_to_completion"].notna()]
df_control_completed = df_control[df_control["avg_time_to_completion"].notna()]

# Calculate average time to completion per group
avg_time_to_completion_test = df_test_completed["avg_time_to_completion"].mean()
avg_time_to_completion_control = df_control_completed["avg_time_to_completion"].mean()

avg_time_to_completion_test, avg_time_to_completion_control

The test group completes the process faster on average (~383s) compared to the control group (~399s), suggesting improved efficiency in the new design.

We now perform statistical testing to determine whether this difference is significant.

### Hypothesis Testing

We test whether the new design affects the time to completion.

- Null hypothesis (H₀): There is no difference in average time to completion between the test and control groups.
- Alternative hypothesis (H₁): The average time to completion differs between the test and control groups.

In [ ]:
# Statistical test: independent t-test

from scipy.stats import ttest_ind

# Extract values
test_times = df_test_completed["avg_time_to_completion"]
control_times = df_control_completed["avg_time_to_completion"]

# Run t-test
t_stat, p_value = ttest_ind(test_times, control_times, equal_var=False)

t_stat, p_value

The test group completes the process faster on average (~383s) compared to the control group (~399s).

The difference is statistically significant (p ≈ 0.002), so we reject the null hypothesis.

This suggests that the new design improves the efficiency of the completion process.

### Error Rate

We analyze the error rate to understand how often users encounter issues during the process.

A lower error rate indicates a smoother and more reliable user experience, while a higher error rate may signal friction or usability problems in the design.

In [ ]:
# Error Rate per group (using backward steps as proxy)
error_rate_test = df_test["n_backward_steps"].mean()
error_rate_control = df_control["n_backward_steps"].mean()

error_rate_test, error_rate_control

The test group shows a higher error rate (~0.61 backward steps per user) compared to the control group (~0.42).

This suggests that users in the test group encounter more friction and need to navigate backward more often during the process.

Statistical testing is required to determine whether this difference is significant.

### Hypothesis Testing – Error Rate

We test whether the error rate differs between the test and control groups.

- Null hypothesis (H₀): There is no difference in error rate between the groups  
- Alternative hypothesis (H₁): The error rate differs between the groups  

An independent t-test was conducted.

The results show a statistically significant difference (p < 0.001).

The test group has a higher error rate than the control group, indicating that users in the new design experience more friction and navigate backward more often.

This suggests a potential usability issue introduced by the new design.

In [ ]:
from scipy.stats import ttest_ind

# Extract values
test_errors = df_test["n_backward_steps"]
control_errors = df_control["n_backward_steps"]

# Run t-test
t_stat_error, p_value_error = ttest_ind(test_errors, control_errors, equal_var=False)

t_stat_error, p_value_error

### Completion Rate

We analyze the completion rate to understand how many users successfully reach the final "confirm" step.

A higher completion rate indicates a more effective and user-friendly process, while a lower rate may suggest friction or drop-offs in the funnel.

In [ ]:
# Completion Rate per group

completion_rate_test = df_clients[df_clients["Variation"] == "Test"]["completion_rate"].mean()
completion_rate_control = df_clients[df_clients["Variation"] == "Control"]["completion_rate"].mean()

completion_rate_test, completion_rate_control

### Hypothesis Testing – Completion Rate

We test whether the completion rate differs between the test and control groups.

- Null hypothesis (H₀): There is no difference in completion rate between the groups  
- Alternative hypothesis (H₁): The completion rate differs between the groups  

An independent t-test was conducted.

The results show a statistically significant difference (p < 0.001).

The test group has a higher completion rate than the control group, indicating that more users successfully reach the final "confirm" step in the new design.

This suggests that the new design improves the effectiveness of the funnel.

In [ ]:
from scipy.stats import ttest_ind

# Extract values
test_completion = df_clients[df_clients["Variation"] == "Test"]["completion_rate"]
control_completion = df_clients[df_clients["Variation"] == "Control"]["completion_rate"]

# Run t-test
t_stat_completion, p_value_completion = ttest_ind(test_completion, control_completion, equal_var=False)

t_stat_completion, p_value_completion